In [ ]:
# installing required libraries
%pip install transformers datasets accelerate -q

SyntaxError: invalid syntax (518612787.py, line 2)

In [ ]:
# imports and setup

import os
import numpy as np
import pandas as pd
import torch

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from datasets import Dataset

/Users/natashaagapova/Documents/! INNOPOLIS/! THESIS/my-repository/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/natashaagapova/Documents/! INNOPOLIS/! THESIS/my-repository/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# data loading and supercategory mapping
BASE_DIR = ".."
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")

df_train = pd.read_csv(os.path.join(PROCESSED_DIR, "train.csv"))
df_val = pd.read_csv(os.path.join(PROCESSED_DIR, "val.csv"))
df_test = pd.read_csv(os.path.join(PROCESSED_DIR, "test.csv"))

# подгружаем supercategory mapping
mapping = pd.read_csv(os.path.join(PROCESSED_DIR, "label_to_supercategory_v1.csv"))
label_to_supercat = dict(zip(mapping["label"], mapping["supercategory"]))

for df in [df_train, df_val, df_test]:
    df["supercategory"] = df["label"].map(label_to_supercat)

print(df_train["supercategory"].value_counts())

supercategory
sysadmin_devops_network     3308
generic_it_ops              2913
it_governance_leadership    2129
project_product             2006
technical_specialized       1982
backend_general_dev         1546
sales_account               1029
tech_support_helpdesk       1016
web_frontend                 601
Name: count, dtype: int64


In [ ]:
# label encoding for supercategory

le = LabelEncoder()

df_train["y"] = le.fit_transform(df_train["supercategory"])
df_val["y"] = le.transform(df_val["supercategory"])
df_test["y"] = le.transform(df_test["supercategory"])

num_labels = len(le.classes_)
print("Num labels:", num_labels)

Num labels: 9


In [ ]:
# load base bert model
from transformers import AutoModel

AutoModel.from_pretrained("bert-base-uncased")

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [ ]:
# tokenizer setup
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# tokenize resume texts
def tokenize(batch):
    return tokenizer(
        batch["resume_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [ ]:
# create torch dataset objects

train_dataset = Dataset.from_pandas(df_train[["resume_text", "y"]])
val_dataset = Dataset.from_pandas(df_val[["resume_text", "y"]])
test_dataset = Dataset.from_pandas(df_test[["resume_text", "y"]])

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column("y", "labels")
val_dataset = val_dataset.rename_column("y", "labels")
test_dataset = test_dataset.rename_column("y", "labels")

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map: 100%|██████████| 5510/5510 [00:07<00:00, 704.14 examples/s]


In [ ]:
# load bert for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# training arguments setup
training_args = TrainingArguments(
    output_dir="./bert_9classes",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,   # fast
    weight_decay=0.01,
    logging_steps=100
)

In [ ]:
# training arguments setup (sqrt rw, 2 epochs)

training_args_rw2 = TrainingArguments(
    output_dir="./bert_9classes_sqrt_rw_2ep",
    learning_rate=training_args.learning_rate,
    per_device_train_batch_size=training_args.per_device_train_batch_size,
    per_device_eval_batch_size=training_args.per_device_eval_batch_size,
    num_train_epochs=2,
    weight_decay=training_args.weight_decay,
    logging_steps=training_args.logging_steps
)

In [ ]:
# evaluation metrics for validation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }

In [ ]:
# training loop with validation set evaluation
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

/Users/natashaagapova/Documents/! INNOPOLIS/! THESIS/my-repository/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
100,2.145200
200,2.055400
300,1.955000
400,1.850200
500,1.761000
600,1.604800
700,1.453000
800,1.398100
900,1.463700
1000,1.311800


TrainOutput(global_step=2067, training_loss=1.4450823004602176, metrics={'train_runtime': 2269.5686, 'train_samples_per_second': 7.283, 'train_steps_per_second': 0.911, 'total_flos': 1087374773675520.0, 'train_loss': 1.4450823004602176, 'epoch': 1.0})

In [ ]:
# evaluate model on test set
predictions = trainer.predict(test_dataset)

y_true_base = predictions.label_ids
y_pred_base = np.argmax(predictions.predictions, axis=1)

print("Baseline Accuracy:", accuracy_score(y_true_base, y_pred_base))
print("Baseline Macro F1:", f1_score(y_true_base, y_pred_base, average="macro"))

/Users/natashaagapova/Documents/! INNOPOLIS/! THESIS/my-repository/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Accuracy: 0.5883847549909256
Macro F1: 0.6042777982508221


In [ ]:
# reweighting by city_group (train only)

# 1) city group counts
city_counts_train = df_train["city_group"].value_counts(dropna=False)

# 2) inverse-frequency weights
# base formula: w_g = 1 / n_g
# then normalize so that the average weight is 1 (more stable optimization)
raw_w = 1.0 / np.sqrt(city_counts_train)
norm_w = raw_w / raw_w.mean()

city_weight_map = norm_w.to_dict()

# 3) assign each object its group weight
df_train_rw = df_train.copy()
df_val_rw = df_val.copy()
df_test_rw = df_test.copy()

df_train_rw["sample_weight"] = df_train_rw["city_group"].map(city_weight_map).astype(float)

# val/test: weight = 1 (we don't "reweight" there, only evaluate)
df_val_rw["sample_weight"] = 1.0
df_test_rw["sample_weight"] = 1.0

print("Train weights summary:")
display(df_train_rw["sample_weight"].describe())

print("\nTop-10 highest-weight cities (rarest groups):")
display(pd.Series(city_weight_map).sort_values(ascending=False).head(10))

print("\nTop-10 lowest-weight cities (most frequent groups):")
display(pd.Series(city_weight_map).sort_values(ascending=True).head(10))

Train weights summary:


count    16530.000000
mean         0.271415
std          0.444703
min          0.019481
25%          0.019481
50%          0.029519
75%          0.374749
max          1.919766
Name: sample_weight, dtype: float64


Top-10 highest-weight cities (rarest groups):


Набережные Челны    1.919766
Кемерово            1.854690
Сочи                1.793880
Тверь               1.736932
Пенза               1.736932
Оренбург            1.736932
Ставрополь          1.709792
Симферополь         1.633234
Ульяновск           1.541221
Калуга              1.541221
dtype: float64


Top-10 lowest-weight cities (most frequent groups):


Москва             0.019481
Other              0.029519
Санкт-Петербург    0.060894
Краснодар          0.253303
Новосибирск        0.303121
Казань             0.309991
Екатеринбург       0.374749
Самара             0.403789
Ростов-на-Дону     0.450316
Нижний Новгород    0.465645
dtype: float64

In [ ]:
# build HuggingFace datasets with sample_weight

train_dataset_rw = Dataset.from_pandas(df_train_rw[["resume_text", "y", "sample_weight"]])
val_dataset_rw   = Dataset.from_pandas(df_val_rw[["resume_text", "y", "sample_weight"]])
test_dataset_rw  = Dataset.from_pandas(df_test_rw[["resume_text", "y", "sample_weight"]])

train_dataset_rw = train_dataset_rw.map(tokenize, batched=True)
val_dataset_rw   = val_dataset_rw.map(tokenize, batched=True)
test_dataset_rw  = test_dataset_rw.map(tokenize, batched=True)

train_dataset_rw = train_dataset_rw.rename_column("y", "labels")
val_dataset_rw   = val_dataset_rw.rename_column("y", "labels")
test_dataset_rw  = test_dataset_rw.rename_column("y", "labels")

train_dataset_rw.set_format("torch", columns=["input_ids", "attention_mask", "labels", "sample_weight"])
val_dataset_rw.set_format("torch", columns=["input_ids", "attention_mask", "labels", "sample_weight"])
test_dataset_rw.set_format("torch", columns=["input_ids", "attention_mask", "labels", "sample_weight"])

print(train_dataset_rw[0].keys())

Map: 100%|██████████| 5510/5510 [00:16<00:00, 326.26 examples/s]

dict_keys(['labels', 'sample_weight', 'input_ids', 'attention_mask'])


In [ ]:
# weighted trainer definition
import torch
from transformers import Trainer

class WeightedTrainer(Trainer):
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,   # <-- added
        **kwargs                   # <-- for compatibility
    ):
        sample_weight = inputs.pop("sample_weight", None)
        labels = inputs.get("labels")

        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
        per_sample_loss = loss_fct(logits, labels)

        if sample_weight is None:
            loss = per_sample_loss.mean()
        else:
            sample_weight = sample_weight.to(per_sample_loss.dtype)
            loss = (per_sample_loss * sample_weight).mean()

        return (loss, outputs) if return_outputs else loss

In [ ]:
# reweighted fine-tuning run

model_rw = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

trainer_rw = WeightedTrainer(
    model=model_rw,
    args=training_args,          # same args, 1 epoch, as baseline
    train_dataset=train_dataset_rw,
    eval_dataset=val_dataset_rw,
    compute_metrics=compute_metrics
)

trainer_rw.train()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
100,2.134700
200,1.999300
300,1.939200
400,1.848400
500,1.689200
600,1.531100
700,1.381500
800,1.306800
900,1.339900
1000,1.224000


TrainOutput(global_step=2067, training_loss=1.3964894612630208, metrics={'train_runtime': 3131.598, 'train_samples_per_second': 5.278, 'train_steps_per_second': 0.66, 'total_flos': 1087374773675520.0, 'train_loss': 1.3964894612630208, 'epoch': 1.0})

In [ ]:
# reweighted fine-tuning run (2 epochs)

model_rw2 = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

trainer_rw2 = WeightedTrainer(
    model=model_rw2,
    args=training_args_rw2,
    train_dataset=train_dataset_rw,
    eval_dataset=val_dataset_rw,
    compute_metrics=compute_metrics
)

trainer_rw2.train()

In [ ]:
# reweighted evaluation

predictions_rw = trainer_rw.predict(test_dataset_rw)

y_true_rw1 = predictions_rw.label_ids
y_pred_rw1 = np.argmax(predictions_rw.predictions, axis=1)

print("Sqrt-Reweighted (1ep) Accuracy:", accuracy_score(y_true_rw1, y_pred_rw1))
print("Sqrt-Reweighted (1ep) Macro F1:", f1_score(y_true_rw1, y_pred_rw1, average="macro"))

/Users/natashaagapova/Documents/! INNOPOLIS/! THESIS/my-repository/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Reweighted Accuracy: 0.593103448275862
Reweighted Macro F1: 0.6074939947491886


In [ ]:
# reweighted evaluation (2 epochs)

predictions_rw2 = trainer_rw2.predict(test_dataset_rw)

y_true_rw2 = predictions_rw2.label_ids
y_pred_rw2 = np.argmax(predictions_rw2.predictions, axis=1)

print("Sqrt-Reweighted (2ep) Accuracy:", accuracy_score(y_true_rw2, y_pred_rw2))
print("Sqrt-Reweighted (2ep) Macro F1:", f1_score(y_true_rw2, y_pred_rw2, average="macro"))

In [ ]:
# robust fairness (city n >= 50)

def filter_groups_by_min_n(df, group_col, min_n=50):
    counts = df[group_col].value_counts(dropna=False)
    keep = counts[counts >= min_n].index
    df_f = df[df[group_col].isin(keep)].copy()
    return df_f, counts, keep

In [ ]:
# sqrt reweighting (1 / sqrt(n))

import numpy as np

def compute_sqrt_inverse_weights(city_series):
    city_counts = city_series.value_counts()
    raw_w = 1.0 / np.sqrt(city_counts)
    norm_w = raw_w / raw_w.mean()
    return norm_w

In [ ]:
# unified evaluation function

from sklearn.metrics import accuracy_score, f1_score

def evaluate_all(df_test, y_true, y_pred, num_classes, min_city_n=50):
    df_eval = df_test.copy()
    df_eval["y_true"] = y_true
    df_eval["y_pred"] = y_pred
    df_eval["correct"] = (df_eval["y_true"] == df_eval["y_pred"])

    out = {}
    out["acc"] = float(accuracy_score(y_true, y_pred))
    out["macro_f1"] = float(f1_score(y_true, y_pred, average="macro"))

    # Accuracy gaps
    out["gap_acc_gender"] = float(gap_by_group(df_eval, "gender"))
    out["gap_acc_age"]    = float(gap_by_group(df_eval, "age_group"))
    out["gap_acc_city"]   = float(gap_by_group(df_eval, "city_group"))

    # OVR gaps (full)
    for col in ["gender", "age_group", "city_group"]:
        tpr, fpr, _, _ = ovr_rates_by_group(df_eval, col, num_classes)
        _, tpr_worst, tpr_macro = summarize_gaps(tpr)
        _, fpr_worst, fpr_macro = summarize_gaps(fpr)
        out[f"{col}_tpr_gap_macro"] = tpr_macro
        out[f"{col}_tpr_gap_worst"] = tpr_worst
        out[f"{col}_fpr_gap_macro"] = fpr_macro
        out[f"{col}_fpr_gap_worst"] = fpr_worst

    # Robust city only
    df_city_rob, city_counts, keep = filter_groups_by_min_n(df_eval, "city_group", min_city_n)
    tpr_r, fpr_r, _, _ = ovr_rates_by_group(df_city_rob, "city_group", num_classes)
    _, tpr_worst_r, tpr_macro_r = summarize_gaps(tpr_r)
    _, fpr_worst_r, fpr_macro_r = summarize_gaps(fpr_r)

    out["city_robust_min_n"] = int(min_city_n)
    out["city_robust_num_groups"] = int(len(keep))
    out["city_robust_rows"] = int(len(df_city_rob))
    out["city_robust_tpr_gap_macro"] = tpr_macro_r
    out["city_robust_tpr_gap_worst"] = tpr_worst_r
    out["city_robust_fpr_gap_macro"] = fpr_macro_r
    out["city_robust_fpr_gap_worst"] = fpr_worst_r

    return out

In [ ]:
# final comparison table (baseline vs sqrt rw)

MIN_CITY_N = 50

res_base = evaluate_all(df_test, y_true_base, y_pred_base, num_labels, min_city_n=MIN_CITY_N)
res_rw1  = evaluate_all(df_test, y_true_rw1,  y_pred_rw1,  num_labels, min_city_n=MIN_CITY_N)
res_rw2  = evaluate_all(df_test, y_true_rw2,  y_pred_rw2,  num_labels, min_city_n=MIN_CITY_N)

results = pd.DataFrame([
    {"model": "baseline_1ep", **res_base},
    {"model": "sqrt_rw_1ep",  **res_rw1},
    {"model": "sqrt_rw_2ep",  **res_rw2},
])

results

In [ ]:
# fairness helper functions (gap/tpr/fpr)

import numpy as np
import pandas as pd


def gap_by_group(df, group_col):
    """
    Accuracy gap:
    max_group_accuracy - min_group_accuracy
    """
    group_acc = (
        df.groupby(group_col)["correct"]
        .mean()
        .dropna()
    )
    
    if len(group_acc) == 0:
        return np.nan
    
    return group_acc.max() - group_acc.min()


def ovr_rates_by_group(df, group_col, num_classes):
    """
    Computes one-vs-rest TPR and FPR for each class and group.

    Returns:
        tpr: array (num_groups, num_classes)
        fpr: array (num_groups, num_classes)
        support_pos: number of positive samples per (group, class)
        support_neg: number of negative samples per (group, class)
    """
    
    groups = sorted(df[group_col].dropna().unique())
    group_index = {g: i for i, g in enumerate(groups)}
    
    tpr = np.zeros((len(groups), num_classes))
    fpr = np.zeros((len(groups), num_classes))
    support_pos = np.zeros((len(groups), num_classes))
    support_neg = np.zeros((len(groups), num_classes))
    
    for g in groups:
        df_g = df[df[group_col] == g]
        gi = group_index[g]
        
        y_true = df_g["y_true"].values
        y_pred = df_g["y_pred"].values
        
        for c in range(num_classes):
            # Positive = class c
            pos_mask = (y_true == c)
            neg_mask = (y_true != c)
            
            TP = np.sum((y_pred == c) & pos_mask)
            FP = np.sum((y_pred == c) & neg_mask)
            FN = np.sum((y_pred != c) & pos_mask)
            TN = np.sum((y_pred != c) & neg_mask)
            
            support_pos[gi, c] = np.sum(pos_mask)
            support_neg[gi, c] = np.sum(neg_mask)
            
            # TPR = TP / (TP + FN)
            if (TP + FN) > 0:
                tpr[gi, c] = TP / (TP + FN)
            else:
                tpr[gi, c] = np.nan
            
            # FPR = FP / (FP + TN)
            if (FP + TN) > 0:
                fpr[gi, c] = FP / (FP + TN)
            else:
                fpr[gi, c] = np.nan
    
    return tpr, fpr, support_pos, support_neg


def summarize_gaps(rate_matrix):
    """
    Given rate matrix (num_groups, num_classes),
    computes:
        - gap per class
        - worst class gap
        - macro gap (mean over classes)
    """
    
    gap_per_class = []
    
    for c in range(rate_matrix.shape[1]):
        col = rate_matrix[:, c]
        col = col[~np.isnan(col)]
        
        if len(col) == 0:
            gap_per_class.append(np.nan)
        else:
            gap_per_class.append(np.max(col) - np.min(col))
    
    gap_per_class = np.array(gap_per_class)
    
    valid = gap_per_class[~np.isnan(gap_per_class)]
    
    if len(valid) == 0:
        return gap_per_class, np.nan, np.nan
    
    worst_gap = np.max(valid)
    macro_gap = np.mean(valid)
    
    return gap_per_class, worst_gap, macro_gap